In [ ]:
# -- COLAB SETUP --
import subprocess
import sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mlcolvar', 'lightning'])

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print('Packages installed and imported successfully.')
print(f'PyTorch version: {torch.__version__}')
print(f'NumPy version: {np.__version__}')

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jolayfield/chem-lab-tutorials/blob/main/Track3_PLUMED/mlcolvar_deeplda_deeptica.ipynb)

# Machine-Learned Collective Variables with mlcolvar: Deep-LDA and Deep-TICA

## Table of Contents

1. What is mlcolvar?
2. Data Preparation for Deep-LDA
3. Deep-LDA: Theory
4. Deep-LDA: Model and Training
5. Deep-LDA: Analysis
6. Deep-TICA: Theory
7. Deep-TICA: Data Preparation
8. Deep-TICA: Model and Training
9. Deep-TICA: Analysis
10. Deep-LDA vs. Deep-TICA: When to Use Each
11. Deploying the Learned CV in PLUMED

## 1. What is mlcolvar?

**mlcolvar** is a unified framework for training machine-learned collective variables (CVs) for use in molecular dynamics simulations, particularly in enhanced sampling methods. It provides PyTorch-based implementations of multiple CV methods and integrates seamlessly with PyTorch Lightning for training.

### Key Features

- **Built on PyTorch + PyTorch Lightning**: Leverages automatic differentiation and distributed training capabilities
- **Modular Architecture**:
  - `mlcolvar.cvs.*`: CV classes (DeepLDA, DeepTICA, AutoEncoderCV, etc.), each a PyTorch Lightning Module
  - `mlcolvar.data.DictDataset` and `DictModule`: data containers using dictionaries of tensors
  - `mlcolvar.core`: neural network building blocks, loss functions (ReduceEigenvaluesLoss), normalizations
  - `mlcolvar.utils`: I/O helpers, time-lagged dataset creation, PLUMED export utilities

### Available CV Methods

| Category | Method | Requires | Output |
|----------|--------|----------|--------|
| Supervised | LDA | labeled data | $n_{states} - 1$ CVs |
| Supervised | DeepLDA | labeled data | $n_{states} - 1$ CVs |
| Supervised | DeepTDA | labeled data | $n_{states} - 1$ CVs |
| Unsupervised | PCA | unlabeled data | $n_{cvs}$ principal components |
| Unsupervised | AutoEncoder | unlabeled data | learned bottleneck |
| Time-informed | TICA | time-lagged trajectory | $n_{cvs}$ slow modes |
| Time-informed | DeepTICA | time-lagged trajectory | $n_{cvs}$ slow modes |
| Time-informed | VDE | time-lagged trajectory | variational dynamics encoder |
| Transition | Committor | committor labels | committor probability |

### Reference

Bonati et al., "Machine learning collective variables for enhanced sampling with rare event techniques", *J. Chem. Phys.* **159**, 014801 (2023)

## Data Generation

We generate synthetic alanine dipeptide data that mimics a real MD simulation. The data contains two metastable states (alpha-helix and beta-sheet) characterized by backbone dihedral angles, plus synthetic interatomic distance features.

### Labeled Dataset for Deep-LDA

We create 4000 frames (2000 per state) with features representing interatomic distances. The states are clearly separated in coordinate space.

### Time-Correlated Trajectory for Deep-TICA

We create 8000 frames of a continuous trajectory with rare transitions between states. The time-correlation structure allows TICA to identify slow modes.

In [ ]:
# Generate synthetic alanine dipeptide data

# ===== STATE DEFINITIONS =====
# Alpha helix state
n_alpha = 2000
phi_alpha = np.random.normal(loc=-1.05, scale=0.20, size=n_alpha)  # radians (~-60 degrees)
psi_alpha = np.random.normal(loc=-0.70, scale=0.20, size=n_alpha)  # radians (~-40 degrees)

# Beta sheet state
n_beta = 2000
phi_beta = np.random.normal(loc=-2.36, scale=0.25, size=n_beta)   # radians (~-135 degrees)
psi_beta = np.random.normal(loc=2.44, scale=0.25, size=n_beta)    # radians (~+140 degrees)

# Combine
phi_labeled = np.concatenate([phi_alpha, phi_beta])
psi_labeled = np.concatenate([psi_alpha, psi_beta])
y_labeled = np.concatenate([np.zeros(n_alpha, dtype=np.int64), np.ones(n_beta, dtype=np.int64)])

# ===== GENERATE DISTANCE FEATURES =====
n_features = 45
n_total = len(phi_labeled)

# Base distances (initial values for each feature in each state)
base_dist_alpha = np.random.uniform(0.3, 1.5, size=n_features)
base_dist_beta = base_dist_alpha + np.random.uniform(-0.3, 0.3, size=n_features)

# Sensitivity weights (how much each feature responds to phi and psi)
w1 = np.random.normal(0, 0.3, size=n_features)  # sensitivity to phi
w2 = np.random.normal(0, 0.3, size=n_features)  # sensitivity to psi

# Compute mean angles (for centering)
mean_phi_alpha = phi_alpha.mean()
mean_psi_alpha = psi_alpha.mean()
mean_phi_beta = phi_beta.mean()
mean_psi_beta = psi_beta.mean()

# Generate distance features
X_labeled = np.zeros((n_total, n_features), dtype=np.float32)

for i in range(n_alpha):
    perturbation_phi = (phi_alpha[i] - mean_phi_alpha)
    perturbation_psi = (psi_alpha[i] - mean_psi_alpha)
    noise = np.random.normal(0, 0.02, size=n_features)
    X_labeled[i, :] = base_dist_alpha + w1 * perturbation_phi + w2 * perturbation_psi + noise

for i in range(n_beta):
    perturbation_phi = (phi_beta[i] - mean_phi_beta)
    perturbation_psi = (psi_beta[i] - mean_psi_beta)
    noise = np.random.normal(0, 0.02, size=n_features)
    X_labeled[n_alpha + i, :] = base_dist_beta + w1 * perturbation_phi + w2 * perturbation_psi + noise

# Clip distances to physical range
X_labeled = np.clip(X_labeled, 0.2, 2.0)

print('Labeled dataset (for Deep-LDA):')
print(f'  X_labeled shape: {X_labeled.shape}, dtype: {X_labeled.dtype}')
print(f'  y_labeled shape: {y_labeled.shape}, dtype: {y_labeled.dtype}')
print(f'  State distribution: {np.bincount(y_labeled)}')
print(f'  Feature range: [{X_labeled.min():.3f}, {X_labeled.max():.3f}] nm')

In [ ]:
# Generate time-correlated trajectory for Deep-TICA
# Using a 2D random walk with occasional jumps between basins

n_frames = 8000
phi_traj = np.zeros(n_frames)
psi_traj = np.zeros(n_frames)

# Start in alpha basin
phi_traj[0] = -1.05
psi_traj[0] = -0.70

step_size = 0.05
jump_interval = 200
jump_prob = 0.3

# Basin centers
alpha_center = np.array([-1.05, -0.70])
beta_center = np.array([-2.36, 2.44])

for t in range(1, n_frames):
    # Random walk step
    phi_traj[t] = phi_traj[t-1] + step_size * np.random.normal()
    psi_traj[t] = psi_traj[t-1] + step_size * np.random.normal()
    
    # Occasional basin jump
    if t % jump_interval == 0 and np.random.rand() < jump_prob:
        current_pos = np.array([phi_traj[t], psi_traj[t]])
        dist_to_alpha = np.linalg.norm(current_pos - alpha_center)
        dist_to_beta = np.linalg.norm(current_pos - beta_center)
        
        if dist_to_alpha < dist_to_beta:
            # Jump to beta
            phi_traj[t] = beta_center[0] + np.random.normal(0, 0.3)
            psi_traj[t] = beta_center[1] + np.random.normal(0, 0.3)
        else:
            # Jump to alpha
            phi_traj[t] = alpha_center[0] + np.random.normal(0, 0.3)
            psi_traj[t] = alpha_center[1] + np.random.normal(0, 0.3)
    
    # Reflecting boundaries at ±pi (optional, for physical realism)
    phi_traj[t] = np.clip(phi_traj[t], -np.pi, np.pi)
    psi_traj[t] = np.clip(psi_traj[t], -np.pi, np.pi)

# Generate distance features from phi/psi trajectory
X_traj = np.zeros((n_frames, n_features), dtype=np.float32)

for i in range(n_frames):
    noise = np.random.normal(0, 0.02, size=n_features)
    X_traj[i, :] = base_dist_alpha + w1 * (phi_traj[i] - phi_traj[0]) + w2 * (psi_traj[i] - psi_traj[0]) + noise

# Clip to physical range
X_traj = np.clip(X_traj, 0.2, 2.0)

print('Time-correlated trajectory (for Deep-TICA):')
print(f'  X_traj shape: {X_traj.shape}, dtype: {X_traj.dtype}')
print(f'  phi_traj shape: {phi_traj.shape}')
print(f'  psi_traj shape: {psi_traj.shape}')
print(f'  Phi range: [{phi_traj.min():.3f}, {phi_traj.max():.3f}] rad')
print(f'  Psi range: [{psi_traj.min():.3f}, {psi_traj.max():.3f}] rad')

## 2. Data Preparation for Deep-LDA

Deep-LDA requires labeled data: each configuration is assigned to one of several metastable states. The key data structure in mlcolvar is the `DictDataset`, which stores data as a dictionary of tensors.

### DictDataset Format

For Deep-LDA training, the dictionary must contain:
- `'data'`: tensor of shape (N, n_features) containing the input features
- `'labels'`: tensor of shape (N,) containing integer state indices (0, 1, ..., n_states-1)

The `DictModule` wrapper handles train/validation splitting and creates PyTorch DataLoaders.

In [ ]:
from mlcolvar.data import DictDataset, DictModule

# Create DictDataset from synthetic labeled data
dataset = DictDataset({
    'data': torch.tensor(X_labeled, dtype=torch.float32),
    'labels': torch.tensor(y_labeled, dtype=torch.long)
})

# DictModule handles train/val split and batch creation
datamodule = DictModule(dataset, lengths=[0.8, 0.2], batch_size=32)

print(f'Total samples: {len(dataset)}')
print(f'Feature dimension: {dataset["data"].shape[1]}')
print(f'Number of states: {len(dataset["labels"].unique())}')
print(f'State labels: {sorted(dataset["labels"].unique().tolist())}')
print(f'Train/val split: {int(len(dataset) * 0.8)} / {int(len(dataset) * 0.2)}')

In [ ]:
# Visualize the raw features and state separation

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: histogram of first 3 features
ax = axes[0]
for feat_idx in range(3):
    feat_alpha = X_labeled[y_labeled == 0, feat_idx]
    feat_beta = X_labeled[y_labeled == 1, feat_idx]
    ax.hist(feat_alpha, bins=30, alpha=0.5, density=True, label=f'State A (feat {feat_idx})')
    ax.hist(feat_beta, bins=30, alpha=0.5, density=True, label=f'State B (feat {feat_idx})')
ax.set_xlabel('Feature Value (nm)')
ax.set_ylabel('Density')
ax.set_title('Distribution of First 3 Distance Features')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Right: Ramachandran plot colored by state
ax = axes[1]
scatter_a = ax.scatter(phi_labeled[y_labeled == 0], psi_labeled[y_labeled == 0],
                       c='blue', alpha=0.5, s=20, label='State A (alpha)')
scatter_b = ax.scatter(phi_labeled[y_labeled == 1], psi_labeled[y_labeled == 1],
                       c='red', alpha=0.5, s=20, label='State B (beta)')
ax.set_xlabel(r'$\phi$ (radians)')
ax.set_ylabel(r'$\psi$ (radians)')
ax.set_title('Ramachandran Plot: State Distribution')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Feature visualization complete.')

## 3. Deep-LDA: Theory

### Fisher's Linear Discriminant Analysis (LDA)

Classical LDA finds a linear projection that maximizes class separation. For data $x_i$ labeled with class $c_i$, define:

- **Between-class scatter matrix**: $S_B = \sum_c N_c (\mu_c - \mu)(\mu_c - \mu)^T$
- **Within-class scatter matrix**: $S_W = \sum_c \sum_{x \in c} (x - \mu_c)(x - \mu_c)^T$

where $\mu_c$ is the mean of class $c$, and $\mu$ is the global mean.

The Fisher ratio is:
$$J(w) = \frac{w^T S_B w}{w^T S_W w}$$

The optimal projection $w$ is the eigenvector of the generalized eigenvalue problem:
$$S_B v = \lambda S_W v$$

The eigenvalue $\lambda$ directly measures the quality of class separation; larger $\lambda$ indicates better discrimination.

### Deep-LDA

Deep-LDA replaces the input $x$ with a learned representation $f_\theta(x)$ from a neural network:

$$\text{LDA eigenvalues on } f_\theta(x) = \text{argmax}_\theta \lambda_1(\theta)$$

The neural network learns a coordinate system in which the LDA separation is optimal. The loss function is:
$$L = -\lambda_1$$

where $\lambda_1$ is the largest eigenvalue of the generalized LDA problem applied to the NN outputs.

### Network Architecture

For $n_{states}$ classes, the output dimension is $n_{states} - 1$ (the rank of $S_B$). For our 2-state system:
- Input: 45 distance features
- Hidden layers: [30, 30]
- Output: 1 collective variable

In [ ]:
# Illustrate classical LDA geometrically on 2D PCA projection

from sklearn.decomposition import PCA

# Project to 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_labeled)

# Compute LDA direction manually
X_a = X_pca[y_labeled == 0]
X_b = X_pca[y_labeled == 1]
mu_a = X_a.mean(axis=0)
mu_b = X_b.mean(axis=0)
mu = X_pca.mean(axis=0)

# Between-class scatter
S_B = len(X_a) * np.outer(mu_a - mu, mu_a - mu) + len(X_b) * np.outer(mu_b - mu, mu_b - mu)

# Within-class scatter
S_W = np.zeros((2, 2))
for x in X_a:
    diff = (x - mu_a).reshape(-1, 1)
    S_W += diff @ diff.T
for x in X_b:
    diff = (x - mu_b).reshape(-1, 1)
    S_W += diff @ diff.T

# Solve generalized eigenvalue problem: S_B v = lambda S_W v
eigenvalues, eigenvectors = np.linalg.eig(np.linalg.inv(S_W) @ S_B)
idx_max = np.argmax(eigenvalues)
lda_direction = eigenvectors[:, idx_max]
lda_direction = lda_direction / np.linalg.norm(lda_direction)
lda_eigenvalue = eigenvalues[idx_max]

# Compute Fisher ratio
fisher_ratio = lda_eigenvalue

# Plot
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(X_pca[y_labeled == 0, 0], X_pca[y_labeled == 0, 1],
          c='blue', alpha=0.5, s=30, label='State A')
ax.scatter(X_pca[y_labeled == 1, 0], X_pca[y_labeled == 1, 1],
          c='red', alpha=0.5, s=30, label='State B')

# Plot LDA direction as an arrow from global mean
arrow_scale = 3
ax.arrow(mu[0], mu[1], arrow_scale * lda_direction[0], arrow_scale * lda_direction[1],
        head_width=0.3, head_length=0.2, fc='green', ec='green', linewidth=2, label='LDA direction')

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_title(f'Classical LDA on 2D PCA Projection\n(Fisher ratio = {fisher_ratio:.2f})')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Classical LDA eigenvalue: {lda_eigenvalue:.4f}')
print(f'LDA direction: {lda_direction}')

## 4. Deep-LDA: Model and Training

We now train a neural network with LDA loss to learn a nonlinear collective variable that separates the states optimally.

In [ ]:
from mlcolvar.cvs import DeepLDA
import lightning as L

n_input = X_labeled.shape[1]  # 45
n_states = 2

# Network architecture: [45] -> [30] -> [30] -> [1]
nodes = [n_input, 30, 30, n_states]

model = DeepLDA(nodes, n_states=n_states)

print('DeepLDA Model Architecture:')
print(model)

# Count trainable parameters
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {n_params:,}')

In [ ]:
from mlcolvar.utils.trainer import MetricsCallback
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

# Set up callbacks
metrics = MetricsCallback()
early_stop = EarlyStopping(
    monitor='valid_loss',
    patience=30,
    min_delta=0.01,
    verbose=True
)

# Create trainer
trainer = L.Trainer(
    max_epochs=200,
    callbacks=[metrics, early_stop],
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
)

# Train
print('Training Deep-LDA model...')
trainer.fit(model, datamodule)

n_epochs = len(metrics.metrics['train_loss'])
print(f'\nTraining completed in {n_epochs} epochs')

In [ ]:
# Plot training and validation loss

fig, ax = plt.subplots(figsize=(10, 5))

train_loss = metrics.metrics['train_loss']
valid_loss = metrics.metrics['valid_loss']

ax.plot(train_loss, label='Train loss', linewidth=2)
ax.plot(valid_loss, label='Validation loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Deep-LDA Training Curves')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final train loss: {train_loss[-1]:.6f}')
print(f'Final valid loss: {valid_loss[-1]:.6f}')

## 5. Deep-LDA: Analysis

Now we analyze the learned collective variable by computing its values on the test set and examining its discriminative power.

In [ ]:
# Evaluate the trained model on all data

model.eval()
with torch.no_grad():
    X_tensor = torch.tensor(X_labeled, dtype=torch.float32)
    cv_values = model(X_tensor).numpy().squeeze()

# Separate by state
cv_A = cv_values[y_labeled == 0]
cv_B = cv_values[y_labeled == 1]

print('Learned CV statistics:')
print(f'State A: mean = {cv_A.mean():.4f}, std = {cv_A.std():.4f}')
print(f'State B: mean = {cv_B.mean():.4f}, std = {cv_B.std():.4f}')
print(f'Separation (|mean_B - mean_A|): {abs(cv_B.mean() - cv_A.mean()):.4f}')

In [ ]:
# Plot CV distributions and compute Fisher ratio

# Fisher ratio = (mean_B - mean_A)^2 / (var_A + var_B)
mean_diff = cv_B.mean() - cv_A.mean()
var_sum = cv_A.var() + cv_B.var()
fisher_ratio_learned = (mean_diff ** 2) / var_sum

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(cv_A, bins=40, alpha=0.6, density=True, label='State A', color='blue')
ax.hist(cv_B, bins=40, alpha=0.6, density=True, label='State B', color='red')

ax.set_xlabel('Learned CV Value')
ax.set_ylabel('Density')
ax.set_title(f'Deep-LDA Collective Variable Distribution\n(Fisher Ratio = {fisher_ratio_learned:.2f})')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Fisher ratio of learned CV: {fisher_ratio_learned:.4f}')
print(f'Overlap at CV = 0: ~{np.sum((cv_A <= 0) & (cv_A >= -0.5)) / len(cv_A) * 100:.1f}% of State A, ~{np.sum((cv_B <= 0) & (cv_B >= -0.5)) / len(cv_B) * 100:.1f}% of State B')

In [ ]:
# Plot Ramachandran diagram colored by learned CV value

fig, ax = plt.subplots(figsize=(10, 8))

scatter = ax.scatter(phi_labeled, psi_labeled, c=cv_values, s=30, cmap='RdBu_r', alpha=0.7)
cbar = plt.colorbar(scatter, ax=ax, label='Learned CV')

# Overlay contour at CV = 0 (decision boundary)
ax.contour(np.linspace(-np.pi, np.pi, 100), np.linspace(-np.pi, np.pi, 100),
          np.zeros((100, 100)), levels=[0], colors='black', linewidths=1, linestyles='--')

ax.set_xlabel(r'$\phi$ (radians)')
ax.set_ylabel(r'$\psi$ (radians)')
ax.set_title('Ramachandran Plot Colored by Learned CV')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Ramachandran visualization complete.')

In [ ]:
# Compute sensitivity of CV with respect to input features
# d(CV)/d(distance_feature) shows which features the NN relies on

model.eval()

# Select representative frames from each state
idx_a = np.where(y_labeled == 0)[0][:100]
idx_b = np.where(y_labeled == 1)[0][:100]

sensitivities_a = []
sensitivities_b = []

for idx in idx_a:
    x = torch.tensor(X_labeled[idx:idx+1], dtype=torch.float32, requires_grad=True)
    cv = model(x)
    cv.backward()
    sensitivities_a.append(x.grad.numpy().squeeze())

for idx in idx_b:
    x = torch.tensor(X_labeled[idx:idx+1], dtype=torch.float32, requires_grad=True)
    cv = model(x)
    cv.backward()
    sensitivities_b.append(x.grad.numpy().squeeze())

# Average absolute sensitivities
avg_sensitivity_a = np.mean(np.abs(sensitivities_a), axis=0)
avg_sensitivity_b = np.mean(np.abs(sensitivities_b), axis=0)

print('Feature sensitivity computed.')
print(f'Top 5 sensitive features for State A: {np.argsort(avg_sensitivity_a)[-5:][::-1]}')
print(f'Top 5 sensitive features for State B: {np.argsort(avg_sensitivity_b)[-5:][::-1]}')

In [ ]:
# Plot feature sensitivities

fig, ax = plt.subplots(figsize=(12, 5))

x_pos = np.arange(n_features)
width = 0.35

ax.bar(x_pos - width/2, avg_sensitivity_a, width, label='State A', alpha=0.8)
ax.bar(x_pos + width/2, avg_sensitivity_b, width, label='State B', alpha=0.8)

ax.set_xlabel('Distance Feature Index')
ax.set_ylabel('|d(CV)/d(feature)| (average)')
ax.set_title('Feature Sensitivity: Which Distances Drive the Learned CV?')
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Feature sensitivity visualization complete.')

## 6. Deep-TICA: Theory

### Time-Lagged Independent Component Analysis (TICA)

TICA identifies slow modes by decorrelating time-lagged representations of data. Given a time series $x(t)$, define the time-lagged covariance:

$$C(\tau) = \left\langle f(x(t)) f(x(t+\tau))^T \right\rangle_t$$

$$C(0) = \left\langle f(x(t)) f(x(t))^T \right\rangle_t$$

where $\langle \cdot \rangle_t$ denotes time average. The TICA eigenvalue problem is:

$$C(\tau) v = \lambda C(0) v$$

The eigenvalue $\lambda_i$ measures the autocorrelation of mode $i$ at lag $\tau$. Large $\lambda_i$ (close to 1) means the mode varies slowly; small $\lambda_i$ means it equilibrates quickly.

The **implied timescale** associated with eigenvalue $\lambda_i$ is:
$$t_i = -\frac{\tau}{\ln|\lambda_i|}$$

### Deep-TICA

Deep-TICA replaces $f(x) = x$ with a neural network $f_\theta(x)$:

$$\text{maximize} \sum_i \lambda_i^2(\theta)$$

The network learns a representation where the slowest dynamics are most prominent. The loss is:
$$L = -\sum_i \lambda_i^2$$

### Why Slow Modes?

In molecular dynamics, slow modes correspond to rare events and state transitions. Enhancing sampling along these modes is more efficient than sampling along fast, equilibrated coordinates.

### Lag Time Selection

The lag time $\tau$ must be chosen carefully:
- Too small: eigenvalues are all close to 1 (no decorrelation information)
- Too large: eigenvalues are all close to 0 (all modes look slow)
- Optimal: $\tau$ should be significantly smaller than the slowest transition time, but large enough to capture meaningful dynamics

A common strategy is to plot eigenvalues vs. lag time and choose a $\tau$ where the spectrum is stable.

In [ ]:
# Illustrate time-lagged data and autocorrelation

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Top-left: time series
ax = axes[0, 0]
t_show = 500
ax.plot(phi_traj[:t_show], linewidth=1, label=r'$\phi(t)$', color='blue')
ax.set_xlabel('Frame')
ax.set_ylabel(r'$\phi$ (radians)')
ax.set_title('Alanine Dipeptide Dihedral Trajectory (first 500 frames)')
ax.grid(alpha=0.3)

# Top-right: lag pairs illustration
ax = axes[0, 1]
lag_demo = 10
n_arrows = 30
for i in range(0, n_arrows):
    ax.arrow(phi_traj[i], psi_traj[i],
            phi_traj[i+lag_demo] - phi_traj[i],
            psi_traj[i+lag_demo] - psi_traj[i],
            head_width=0.05, head_length=0.05, fc='gray', ec='gray', alpha=0.3)
ax.scatter(phi_traj[:n_arrows+lag_demo], psi_traj[:n_arrows+lag_demo], c='blue', s=20, alpha=0.5)
ax.set_xlabel(r'$\phi$')
ax.set_ylabel(r'$\psi$')
ax.set_title(f'Time-Lagged Pairs on Ramachandran (lag={lag_demo})')
ax.grid(alpha=0.3)

# Bottom-left: autocorrelation of phi
ax = axes[1, 0]
lags = np.arange(0, 200)
acf_phi = np.correlate(phi_traj - phi_traj.mean(), phi_traj - phi_traj.mean(), mode='full')
acf_phi = acf_phi[len(acf_phi)//2:]
acf_phi = acf_phi / acf_phi[0]  # normalize
ax.plot(lags, acf_phi[:len(lags)], linewidth=2, color='blue')
ax.axhline(0, color='k', linestyle='-', linewidth=0.5)
ax.axhline(np.exp(-1), color='r', linestyle='--', linewidth=1, label=r'$e^{-1}$')
ax.set_xlabel('Lag (frames)')
ax.set_ylabel(r'$\rho(\tau)$')
ax.set_title(r'Autocorrelation of $\phi$')
ax.legend()
ax.grid(alpha=0.3)

# Bottom-right: psi autocorrelation
ax = axes[1, 1]
acf_psi = np.correlate(psi_traj - psi_traj.mean(), psi_traj - psi_traj.mean(), mode='full')
acf_psi = acf_psi[len(acf_psi)//2:]
acf_psi = acf_psi / acf_psi[0]
ax.plot(lags, acf_psi[:len(lags)], linewidth=2, color='red')
ax.axhline(0, color='k', linestyle='-', linewidth=0.5)
ax.axhline(np.exp(-1), color='r', linestyle='--', linewidth=1, label=r'$e^{-1}$')
ax.set_xlabel('Lag (frames)')
ax.set_ylabel(r'$\rho(\tau)$')
ax.set_title(r'Autocorrelation of $\psi$')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Estimate correlation time (where acf drops to e^-1)
idx_e_inv_phi = np.argmin(np.abs(acf_phi[:len(lags)] - np.exp(-1)))
print(f'Approximate correlation time for phi: {lags[idx_e_inv_phi]} frames')

## 7. Deep-TICA: Data Preparation

For Deep-TICA training, we need to prepare time-lagged pairs of configurations. The mlcolvar utility `create_timelagged_dataset` handles this automatically.

In [ ]:
from mlcolvar.utils.timelagged import create_timelagged_dataset

# Choose lag time based on trajectory correlation structure
# Should be large enough to capture slow dynamics but small compared to transition time
lag_time = 10

# Create time-lagged dataset
X_traj_tensor = torch.tensor(X_traj, dtype=torch.float32)
traj_dataset = create_timelagged_dataset(X_traj_tensor, lag_time=lag_time)

print(f'Time-lagged dataset configuration:')
print(f'  Lag time: {lag_time} frames')
print(f'  Total pairs: {len(traj_dataset)}')
print(f'  Keys in first sample: {list(traj_dataset[0].keys())}')

# Inspect shapes
sample = traj_dataset[0]
print(f'  data shape: {sample["data"].shape}')
print(f'  data_lag shape: {sample["data_lag"].shape}')

# Create DictModule for training
traj_datamodule = DictModule(traj_dataset, lengths=[0.8, 0.2], batch_size=32)

print(f'\nDataModule created with 80/20 train/val split')

In [ ]:
# Inspect the time-lagged dataset structure

# Show a few examples
print('First 5 time-lagged pairs (first feature only):')
print('Frame t | Frame t+lag')
for i in range(5):
    sample = traj_dataset[i]
    d_t = sample['data'][0].item()
    d_tlag = sample['data_lag'][0].item()
    print(f'{d_t:.4f}  | {d_tlag:.4f}')

print('\nNote: In biased simulations, weights would also be provided via')
print('  sample["weights"] = exp(-V_bias / kT)')
print('to reweight the TICA covariances and account for biasing potentials.')

## 8. Deep-TICA: Model and Training

Now we train a Deep-TICA model to extract slow collective variables from the time-correlated trajectory.

In [ ]:
from mlcolvar.cvs import DeepTICA

n_input_tica = X_traj.shape[1]  # 45 distance features
n_cvs_tica = 2  # extract 2 slow modes

# Network architecture
nodes_tica = [n_input_tica, 30, 30, n_cvs_tica]

model_tica = DeepTICA(nodes_tica, n_cvs=n_cvs_tica)

print('Deep-TICA Model Architecture:')
print(model_tica)

# Count parameters
n_params_tica = sum(p.numel() for p in model_tica.parameters() if p.requires_grad)
print(f'\nTrainable parameters: {n_params_tica:,}')

In [ ]:
# Train Deep-TICA model

metrics_tica = MetricsCallback()
early_stop_tica = EarlyStopping(
    monitor='valid_loss',
    patience=30,
    min_delta=0.01,
    verbose=True
)

trainer_tica = L.Trainer(
    max_epochs=200,
    callbacks=[metrics_tica, early_stop_tica],
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=True,
)

print('Training Deep-TICA model...')
trainer_tica.fit(model_tica, traj_datamodule)

n_epochs_tica = len(metrics_tica.metrics['train_loss'])
print(f'\nTraining completed in {n_epochs_tica} epochs')

In [ ]:
# Plot training curves

fig, ax = plt.subplots(figsize=(10, 5))

train_loss_tica = metrics_tica.metrics['train_loss']
valid_loss_tica = metrics_tica.metrics['valid_loss']

ax.plot(train_loss_tica, label='Train loss', linewidth=2)
ax.plot(valid_loss_tica, label='Validation loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Deep-TICA Training Curves')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final train loss: {train_loss_tica[-1]:.6f}')
print(f'Final valid loss: {valid_loss_tica[-1]:.6f}')

## 9. Deep-TICA: Analysis

We now evaluate the trained Deep-TICA model and examine the learned slow modes.

In [ ]:
# Evaluate on full trajectory

model_tica.eval()
with torch.no_grad():
    tica_cvs = model_tica(X_traj_tensor).numpy()

tica_cv1 = tica_cvs[:, 0]
tica_cv2 = tica_cvs[:, 1]

print(f'Slow CV 1: min = {tica_cv1.min():.4f}, max = {tica_cv1.max():.4f}, std = {tica_cv1.std():.4f}')
print(f'Slow CV 2: min = {tica_cv2.min():.4f}, max = {tica_cv2.max():.4f}, std = {tica_cv2.std():.4f}')

In [ ]:
# Plot slow CVs on Ramachandran diagram

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CV1 vs Ramachandran
ax = axes[0]
scatter = ax.scatter(phi_traj, psi_traj, c=tica_cv1, s=30, cmap='viridis', alpha=0.7)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Deep-TICA CV1')
ax.set_xlabel(r'$\phi$ (radians)')
ax.set_ylabel(r'$\psi$ (radians)')
ax.set_title('Slow Mode 1 (Deep-TICA)')
ax.grid(alpha=0.3)

# CV2 vs Ramachandran
ax = axes[1]
scatter = ax.scatter(phi_traj, psi_traj, c=tica_cv2, s=30, cmap='plasma', alpha=0.7)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Deep-TICA CV2')
ax.set_xlabel(r'$\phi$ (radians)')
ax.set_ylabel(r'$\psi$ (radians)')
ax.set_title('Slow Mode 2 (Deep-TICA)')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('Slow modes visualization complete.')

In [ ]:
# Compute TICA eigenvalues from the learned CVs
# Eigenvalues measure the autocorrelation of each mode at the chosen lag time

# Project trajectory to learned CV space
cv_space = tica_cvs

# Compute C(0) and C(tau) for the projected CVs
lag = lag_time

# C(0): standard covariance
C0 = np.cov(cv_space.T)

# C(tau): time-lagged covariance
cv_lag = cv_space[:-lag]  # x(t)
cv_future = cv_space[lag:]  # x(t+tau)
Ctau = np.cov(cv_lag.T, cv_future.T)[:n_cvs_tica, n_cvs_tica:]  # cross-covariance

# Solve generalized eigenvalue problem: C(tau) @ v = lambda * C(0) @ v
eigenvalues_tica, _ = np.linalg.eig(np.linalg.inv(C0) @ Ctau)
eigenvalues_tica = np.real(eigenvalues_tica)  # take real part
eigenvalues_tica = np.sort(eigenvalues_tica)[::-1]  # descending order

# Compute implied timescales
implied_timescales = -lag_time / np.log(np.abs(eigenvalues_tica))

print('TICA Eigenvalues and Implied Timescales (lag_time={} frames):'.format(lag_time))
for i, (evals, t_i) in enumerate(zip(eigenvalues_tica, implied_timescales)):
    print(f'  Mode {i+1}: lambda = {evals:.4f}, t_i = {t_i:.1f} frames')

In [ ]:
# Plot eigenvalue spectrum

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Eigenvalues
ax = axes[0]
modes = np.arange(1, len(eigenvalues_tica) + 1)
ax.bar(modes, eigenvalues_tica, alpha=0.7, color='steelblue')
ax.set_xlabel('Mode Index')
ax.set_ylabel(r'Eigenvalue $\lambda$')
ax.set_title(f'TICA Eigenvalue Spectrum (lag={lag_time})')
ax.grid(alpha=0.3, axis='y')

# Implied timescales
ax = axes[1]
ax.bar(modes, implied_timescales, alpha=0.7, color='coral')
ax.set_xlabel('Mode Index')
ax.set_ylabel('Implied Timescale (frames)')
ax.set_title('Implied Timescales')
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('Eigenvalue spectrum visualization complete.')

In [ ]:
# Lag-time sensitivity: robustness of eigenvalues across different lag times
# This shows whether the identified slow modes are genuine or artifacts of lag choice
# Note: This cell may take 1-2 minutes to run

lag_times = [5, 10, 20, 40]
first_eigenvalues = []

print('Computing lag-time sensitivity (may take ~1-2 min)...')

for lag_test in lag_times:
    # Create dataset with this lag
    test_dataset = create_timelagged_dataset(X_traj_tensor, lag_time=lag_test)
    test_datamodule = DictModule(test_dataset, lengths=[0.8, 0.2], batch_size=32)
    
    # Train a fresh model with fewer epochs for speed
    test_model = DeepTICA([n_input_tica, 30, 30, n_cvs_tica], n_cvs=n_cvs_tica)
    test_metrics = MetricsCallback()
    test_early_stop = EarlyStopping(monitor='valid_loss', patience=20, min_delta=0.01)
    
    test_trainer = L.Trainer(
        max_epochs=100,
        callbacks=[test_metrics, test_early_stop],
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
    )
    
    test_trainer.fit(test_model, test_datamodule)
    
    # Compute first eigenvalue
    test_model.eval()
    with torch.no_grad():
        test_cvs = test_model(X_traj_tensor).numpy()
    
    test_cv_lag = test_cvs[:-lag_test]
    test_cv_future = test_cvs[lag_test:]
    test_C0 = np.cov(test_cv_lag.T)
    test_Ctau = np.cov(test_cv_lag.T, test_cv_future.T)[:n_cvs_tica, n_cvs_tica:]
    test_evals = np.linalg.eigvals(np.linalg.inv(test_C0) @ test_Ctau)
    test_evals = np.real(np.sort(test_evals)[::-1])
    
    first_eigenvalues.append(test_evals[0])
    print(f'  Lag {lag_test}: first eigenvalue = {test_evals[0]:.4f}')

print('Lag-time sensitivity analysis complete.')

In [ ]:
# Plot lag-time sensitivity

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(lag_times, first_eigenvalues, 'o-', linewidth=2, markersize=8, color='steelblue')
ax.set_xlabel('Lag Time (frames)')
ax.set_ylabel(r'First TICA Eigenvalue $\lambda_1$')
ax.set_title('Robustness of Learned Slow Modes Across Lag Times')
ax.grid(alpha=0.3)
ax.set_xticks(lag_times)

plt.tight_layout()
plt.show()

print('Interpretation: A flat curve indicates the identified slow modes are robust and')
print('not an artifact of the lag-time choice. Oscillating curves suggest sensitivity')
print('to lag-time and may indicate the need for longer simulation or fine-tuning.')

## 10. Deep-LDA vs. Deep-TICA: When to Use Each

Both Deep-LDA and Deep-TICA learn nonlinear collective variables, but they answer different questions and require different types of data.

### Comparison Table

| Aspect | Deep-LDA | Deep-TICA |
|--------|----------|----------|
| **Data Required** | Labeled configurations from 2+ states | Continuous time-correlated trajectory |
| **Physical Basis** | Fisher discriminant: maximize state separation | Eigenvalue of TICA: identify slowest modes |
| **When to Use** | States are well-defined, can collect unbiased samples | No discrete state labels, time-series available |
| **Output CVs** | $n_{states} - 1$ | $n_{cvs}$ (user-chosen) |
| **Handles Bias** | Yes (reweight labels via importance weights) | Yes (reweight covariances via $\exp(-V_{bias}/kT)$) |
| **Limitation** | Requires prior knowledge of metastable states | Requires sufficient sampling of slow dynamics |
| **Implementation Complexity** | Moderate (LDA is well-understood) | Moderate (spectral decomposition required) |
| **Typical Simulation** | Enhanced sampling after initial exploration | Exploratory phase to find new CVs blindly |

### Hybrid Approaches

mlcolvar also supports multitask and semi-supervised learning:
- **Multitask**: Train Deep-LDA and Deep-TICA losses simultaneously if you have both labeled states and trajectory data
- **Semi-supervised**: Use unbiased trajectory segments for TICA and labeled subsets for LDA to improve CV generalization

These advanced methods are documented in the mlcolvar tutorials but not covered in depth here.

## 11. Deploying the Learned CV in PLUMED

The ultimate goal is to use the learned collective variable in PLUMED for enhanced sampling. mlcolvar models can be exported to TorchScript format (.ptc) and loaded directly in PLUMED 2.9+ using the `PYTORCH_MODEL` action.

### Workflow

1. Train a Deep-LDA or Deep-TICA model in this notebook
2. Export it as TorchScript (.ptc file)
3. Write a PLUMED input file that loads the model via `PYTORCH_MODEL`
4. Compute the distance features in PLUMED (same features used for training)
5. Bias along the learned CV using OPES_METAD or other methods

The advantage: the neural network CV acts exactly like any other PLUMED collective variable, enabling hybrid enhanced sampling strategies.

In [ ]:
# Export the trained Deep-LDA model for use in PLUMED

import os

# Create output directory if it doesn't exist
os.makedirs('data', exist_ok=True)

# Try to use mlcolvar's native export if available; fall back to torch.jit
try:
    model.to_plumed('data/deeplda_cv.ptc')
    print('Model exported using mlcolvar.to_plumed()')
except (AttributeError, NotImplementedError):
    # Fallback: use torch.jit.script
    scripted = torch.jit.script(model)
    scripted.save('data/deeplda_cv.ptc')
    print('Model exported using torch.jit.script()')

file_size = os.path.getsize('data/deeplda_cv.ptc') / 1024
print(f'File size: {file_size:.1f} kB')
print('\nModel is ready to load in PLUMED 2.9+ with:')
print('  cv: PYTORCH_MODEL FILE=deeplda_cv.ptc ARG=d')

In [ ]:
# Generate example PLUMED input file using the learned CV

plumed_input = """# plumed_deeplda.dat
# Use a Deep-LDA learned collective variable with OPES_METAD for enhanced sampling
# of alanine dipeptide conformational transitions

MOLINFO STRUCTURE=ala_dipeptide.pdb

# Compute the 45 interatomic distances that served as input features during training
# (Replace with your actual atom pair definitions)
d1: DISTANCE ATOMS=1,2
d2: DISTANCE ATOMS=1,3
d3: DISTANCE ATOMS=2,3
# ... (44 more distances to reach 45 total)

# Aggregate distances into a single argument vector (comma-separated)
DISTANCES: COMBINE ARG=d1,d2,d3,...,d45

# Load the trained Deep-LDA neural network CV
# Inputs: the 45 normalized distances
# Output: 1 collective variable (LDA projection)
cv_lda: PYTORCH_MODEL FILE=deeplda_cv.ptc ARG=DISTANCES

# Apply OPES_METAD along the learned CV
# BARRIER: estimated free energy barrier (in kJ/mol)
# SIGMA: width of deposited Gaussians
# PACE: frequency of bias potential updates
opes: OPES_METAD ARG=cv_lda PACE=500 BARRIER=40 SIGMA=0.05

# Print trajectory
PRINT ARG=cv_lda,opes.bias,opes.rct FILE=COLVAR STRIDE=100
""".strip()

print('Example PLUMED input file:')
print('=' * 70)
print(plumed_input)
print('=' * 70)

# Write to file
os.makedirs('data', exist_ok=True)
with open('data/plumed_deeplda.dat', 'w') as f:
    f.write(plumed_input)

print('\nSaved to data/plumed_deeplda.dat')
print('\nUsage in MD: plumed driver --plumed plumed_deeplda.dat --xtc trajectory.xtc')

In [ ]:
# Summary and practical notes

print('=' * 70)
print('SUMMARY: Deploying Machine-Learned CVs with PLUMED')
print('=' * 70)

print('\n1. TRAINING (this notebook):')
print('   - Collect MD data (labeled for Deep-LDA, trajectories for Deep-TICA)')
print('   - Train neural network with mlcolvar')
print('   - Evaluate and validate the learned CV')

print('\n2. EXPORT:')
print('   - Save model as TorchScript (.ptc): model.to_plumed() or torch.jit.script()')
print('   - File size ~100 kB for typical 2-3 layer networks')

print('\n3. PLUMED INTEGRATION:')
print('   - Compute input features (distances, contacts, etc.) in PLUMED')
print('   - Load model: PYTORCH_MODEL FILE=model.ptc ARG=features')
print('   - Bias the output: OPES_METAD ARG=cv, METAD ARG=cv, etc.')

print('\n4. PRODUCTION SIMULATION:')
print('   - Run MD with PLUMED and the neural network CV')
print('   - Monitor convergence via COLVAR output')
print('   - Reweight bias potential to recover unbiased ensemble (e.g., with plumed sum_hills)')

print('\n5. VALIDATION:')
print('   - Check FE landscape shape: should show clear basins at state A and B')
print('   - Verify transitions occur: monitor time between state crossings')
print('   - Cross-validate with other CVs: compute committor probability')

print('\n' + '=' * 70)
print('Further Reading:')
print('=' * 70)
print('- mlcolvar documentation: https://mlcolvar.github.io/')
print('- PLUMED PYTORCH_MODEL: https://plumed.github.io/doc-v2.9/user-doc/html/PYTORCH_MODEL.html')
print('- Enhanced Sampling: Valsson et al., WCMS 2022')
print('- Neural Network CVs: Bonati et al., JCP 2023')